In [2]:
import os

# Must be set *before* the CUDA context is created (i.e. before the first tensor
# is moved to the GPU) -- reduces fragmentation when encoders are loaded/unloaded
# repeatedly. Takes effect after a kernel restart.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import pickle

import pandas as pd
from sentence_transformers import SentenceTransformer, util
import numpy as np
import json
import importlib
from pathlib import Path
import requests
from transformers import AutoModel, AutoTokenizer, AutoModelForMaskedLM
from transformers import PreTrainedTokenizer
from tensor2tensor.data_generators import text_encoder
from sklearn.decomposition import PCA
from sklearn.manifold      import TSNE
#import umap
import tempun
import torch

from datetime import datetime, timedelta

2026-10-10 21:41:32.935924: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-10 21:41:32.985118: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
outdir = Path("../data/large_files/phylosen_instances/")

In [4]:
passio_df = pd.read_parquet("../data/large_files/passio_sample2000_classified.parquet")
pathos_df = pd.read_parquet("../data/large_files/pathos_sample2000_classified.parquet")

In [5]:
base_path = "/srv/models/latin-bert"

# Initialize the tokenizer with the vocab.txt file and the encoder
vocab_file_path = "/srv/models/latin-bert/vocab.txt" # "/Users/vojtechkase/Projects/latin-bert/models/latin_bert/vocab.txt"  # Update this path
subword_tokenizer_path = "/srv/models/latin-bert/latin.subword.encoder"
# Update this path
encoder = text_encoder.SubwordTextEncoder(subword_tokenizer_path)

spec = importlib.util.spec_from_file_location("latin_tokenizer", os.path.join(base_path, "latin_tokenizer.py"))
latin_tokenizer = importlib.util.module_from_spec(spec)
loader = importlib.util.LazyLoader(spec.loader)
spec.loader.exec_module(latin_tokenizer)

LatinTokenizer = latin_tokenizer.LatinTokenizer

tokenizer_labert = LatinTokenizer(vocab_file_path, encoder)
# model_labert = AutoModel.from_pretrained(base_path)

model_labert_wsd = AutoModel.from_pretrained(
    "../../labyrinth/data/models/latin-bert-wic-ft",
    output_hidden_states=True,
    output_attentions=False
)

In [6]:
tokenizer_philberta = AutoTokenizer.from_pretrained('bowphs/PhilBerta')
model_philberta = AutoModelForMaskedLM.from_pretrained('bowphs/PhilBerta')

In [7]:
tokenizer_xlmr = AutoTokenizer.from_pretrained("xlm-roberta-base")
# output_attentions=False: attentions are never used downstream and cost a lot of
# activation memory (12 layers x 12 heads x L x L) -- a needless OOM risk at L=512.
model_xlmr = AutoModel.from_pretrained(
    "xlm-roberta-base",
    output_hidden_states=True,
    output_attentions=False,
)

In [8]:
tokenizer_mdeberta = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base")
model_mdeberta = AutoModel.from_pretrained(
    "microsoft/mdeberta-v3-base",
    output_hidden_states=True,
    output_attentions=False,  # attentions unused downstream; saves activation memory
)

/home/jupyter-vojta/notebooks/phylosen_pilot/phylosen_venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [9]:
def read_hits(fname):
    try:
        path = os.path.join(outdir, fname)
        instances = pd.read_parquet(path)
    except:
        instances = pd.DataFrame()
    return instances

In [10]:
instances = read_hits("pathos.parquet")
len(instances)

16280

In [11]:
instances.sample(10, random_state=42)

,target_phrase,target_from,matched_by,target_lemmata,target_token_ids,target_kwic_idx,target_sent_idx,grela_id,target_sentence_id,start_sentence_id,end_sentence_id,kwic_text,kwic_tokens,target_sentence_text,target_sentence_tokens,author,title,not_before,not_after
13633,[πάθη],canonical,lemma,[πάθος],[420187460],[10],[4],lagt_tlg0086.tlg041,lagt_tlg0086.tlg041_367,lagt_tlg0086.tlg041_366,lagt_tlg0086.tlg041_367,αὑτὸ συνεχὲς εἰς πεπερασμένα εἴδη . ἐπεὶ οὖν τ...,"[{'token_id': 420187450, 'token_text': 'αὑτὸ',...","ἐπεὶ οὖν τὰ μὲν πάθη ὡς εἴδη λεκτέον , ὑπάρχει...","[{'token_id': 420187456, 'token_text': 'ἐπεὶ',...",Aristotle,De sensu et sensibilibus,-346.0,-334.0
1921,[πάθεσι],canonical,lemma,[πάθος],[432928253],[10],[6],lagt_tlg4102.tlg008,lagt_tlg4102.tlg008_4567,lagt_tlg4102.tlg008_4566,lagt_tlg4102.tlg008_4567,ἡ πεῖρα ἐδίδαξεν · ἔδει γὰρ αὐτὸν τέως δοκιμασ...,"[{'token_id': 432928243, 'token_text': 'ἡ', 'l...","ἔδει γὰρ αὐτὸν τέως δοκιμασθῆναι τοῖς πάθεσι ,...","[{'token_id': 432928247, 'token_text': 'ἔδει',...",Catenae (Novum Testamentum),Catena In Acta,501.0,600.0
12140,[πάθος],canonical,lemma,[πάθος],[415913818],[10],[15],lagt_tlg0057.tlg067,lagt_tlg0057.tlg067_944,lagt_tlg0057.tlg067_944,lagt_tlg0057.tlg067_944,εἰς ὄγκον ἀρθέντος τοῦ μορίου σκοπεῖν εἴ τε φλ...,"[{'token_id': 415913808, 'token_text': 'εἰς', ...",ἐξ ἄλλης δέ τινος αἰτίας εἰς ὄγκον ἀρθέντος το...,"[{'token_id': 415913803, 'token_text': 'ἐξ', '...",Galen,Ad Glauconem de methodo medendi,169.0,175.0
9933,[πάθος],canonical,lemma,[πάθος],[429870222],[10],[8],lagt_tlg0057.tlg076,lagt_tlg0057.tlg076_7741,lagt_tlg0057.tlg076_7740,lagt_tlg0057.tlg076_7741,ἄνθρωπος . ἔτι δὲ σαφέστερον ὁ Διοκλῆς ἡρμήνευ...,"[{'token_id': 429870212, 'token_text': 'ἄνθρωπ...",ἔτι δὲ σαφέστερον ὁ Διοκλῆς ἡρμήνευκεν ὁποῖόν ...,"[{'token_id': 429870214, 'token_text': 'ἔτι', ...",Galen,De compositione medicamentorum secundum locos ...,169.0,175.0
7745,[πάθος],canonical,lemma,[πάθος],[412261397],[10],[28],lagt_tlg0732.tlg007,lagt_tlg0732.tlg007_872,lagt_tlg0732.tlg007_872,lagt_tlg0732.tlg007_873,τὴν τοὐ ὑπαληλιμμένου χρώματος αὐτῷ διαφορὰν κ...,"[{'token_id': 412261387, 'token_text': 'τὴν', ...",οὐ γὰρ ὡς ἁπλοῦν ἔτι διατίθησι τὸ μεταξὺ διαφα...,"[{'token_id': 412261369, 'token_text': 'οὐ', '...",Alexander of Aphrodisias,In Librum De Sensu Commentarium,175.0,225.0
13499,[πάθος],canonical,lemma,[πάθος],[414615864],[10],[6],lagt_tlg4031.tlg002,lagt_tlg4031.tlg002_3081,lagt_tlg4031.tlg002_3080,lagt_tlg4031.tlg002_3082,αὐτῷ τῷ ἐπιθυμεῖν . διὸ καὶ ἀτόπῳ φησὶν ἐοικέν...,"[{'token_id': 414615854, 'token_text': 'αὐτῷ',...",διὸ καὶ ἀτόπῳ φησὶν ἐοικέναι τὸ πάθος αὐτοῦ τὸ...,"[{'token_id': 414615858, 'token_text': 'διὸ', ...",Eustratius,In Aristotelis Ethica Nicomachea I Commentaria,1050.0,1117.0
5046,[πάθεσιν],canonical,lemma,[πάθος],[414739073],[10],[11],lagt_tlg4031.tlg002,lagt_tlg4031.tlg002_8153,lagt_tlg4031.tlg002_8153,lagt_tlg4031.tlg002_8153,γὰρ οἱ ἀκρατεῖς καὶ οἱ προπετεῖς ἐπεὶ ἀπροβουλ...,"[{'token_id': 414739063, 'token_text': 'γὰρ', ...",εἰ γὰρ οἱ ἀκρατεῖς καὶ οἱ προπετεῖς ἐπεὶ ἀπροβ...,"[{'token_id': 414739062, 'token_text': 'εἰ', '...",Eustratius,In Aristotelis Ethica Nicomachea I Commentaria,1050.0,1117.0
10062,[πάθη],canonical,lemma,[πάθος],[415114377],[10],[5],lagt_tlg1264.tlg001,lagt_tlg1264.tlg001_2657,lagt_tlg1264.tlg001_2656,lagt_tlg1264.tlg001_2657,αὐτὴν εἴναι τὴν οὐσίαν · τὰ δ ἄλλα πάντα οἷον ...,"[{'token_id': 415114367, 'token_text': 'αὐτὴν'...","τὰ δ ἄλλα πάντα οἷον πάθη ταύτης , καὶ πὼς ἔχο...","[{'token_id': 415114372, 'token_text': 'τὰ', '...",Chrysippus,Fragmenta Logica et Physica,-279.0,-205.0
8612,[πάθος],canonical,lemma,[πάθος],[420045641],[10],[2],lagt_tlg2733.tlg001,lagt_tlg2733.tlg001_1871,lagt_tlg2733.tlg001_1870,lagt_tlg2733.tlg001_1872,"ἄνευ , καὶ τὸν ἐξ αὐτῆς προελθόντα . Τὸ δὲ πάθ...","[{'token_id': 420045631, 'token_text': 'ἄνευ',...",Τὸ δὲ πάθος ἐκ διαφόρων νοσημάτων συνέκειτο .,"[{'token_id': 420045639, 'token_text': 'Τὸ', '...","Evagrius, Scholasticus",Histori

In [12]:
import embedding_helpers
importlib.reload(embedding_helpers)
from embedding_helpers import *

In [13]:
sample_instance = instances.iloc[0]

In [14]:
# compute embedding
result = embedding_helpers.embed_emlap_instance(
    instance=sample_instance,
    tokenizer=tokenizer_labert,
    model=model_labert_wsd,
    device="cuda",                # or "cpu"
    layer_idx=11,
    piece_pooling="mean",
    context_lemmatized=True,
    target_lemmatized=False,
    context_pos_filtered=True,
)

In [15]:
# compute embedding
result = embedding_helpers.embed_emlap_instance(
    instance=sample_instance,
    tokenizer=tokenizer_philberta,
    model=model_philberta,
    device="cuda",                # or "cpu"
    layer_idx=11,
    piece_pooling="mean",
    context_lemmatized=True,
    target_lemmatized=True,
    context_pos_filtered=True,
)

In [16]:
# compute embedding
result = embedding_helpers.embed_emlap_instance(
    instance=sample_instance,
    tokenizer=tokenizer_mdeberta,
    model=model_mdeberta,
    device="cuda",                # or "cpu"
    layer_idx=11,
    piece_pooling="mean",
    context_lemmatized=True,
    target_lemmatized=True,
    context_pos_filtered=True,
)

In [17]:
model_list = [
        ("labert_wsd", model_labert_wsd, tokenizer_labert),
        ("philberta", model_philberta, tokenizer_philberta),
        ("xlmr", model_xlmr, tokenizer_xlmr),
        ("mdeberta", model_mdeberta, tokenizer_mdeberta),
    ]

In [18]:
#passio_df = pd.read_parquet("../data/large_files/passio_sample2000_classified.parquet")
#pathos_df = pd.read_parquet("../data/large_files/pathos_sample2000_classified.parquet")

# NOTE: encoder-major loop, with each encoder offloaded to CPU before the next
# one is used. Keeping all four encoders resident on a 16 GiB GPU is what causes
# "CUDA out of memory"; `offload()` + `empty_cache()` prevents that.
datasets = {
    "pathos_sample2000_classified_embeddings.parquet": pathos_df,
    "passio_sample2000_classified_embeddings.parquet": passio_df,
}

LAYERS_SAMPLE = {"mdeberta": [6, 7, 8, 9, 10, 11]}  # default for others: [7, 11]

print(f"leftover models on GPU: {[t for t, m, _ in model_list if next(m.parameters()).device.type == 'cuda']}")
embedding_helpers.gpu_mem_report("before sample embeddings")

for title, model, tokenizer in model_list:
    layers = LAYERS_SAMPLE.get(title, [7, 11])
    for layer in layers:
        emb_col_name = f"emb_{title}_l{layer}"
        for df in datasets.values():
            df[emb_col_name] = df.apply(lambda row: embedding_helpers.embed_emlap_instance(
                instance=row,
                tokenizer=tokenizer,
                model=model,
                device="cuda",                # or "cpu"
                layer_idx=layer,
                piece_pooling="mean",
                context_lemmatized=True,
                target_lemmatized=True,
                context_pos_filtered=True)["embedding"], axis=1)
    embedding_helpers.offload(model)          # free this encoder before the next one
    embedding_helpers.gpu_mem_report(f"after offload {title}")

for fn, df in datasets.items():
    df.to_parquet("../data/large_files/{}".format(fn), index=False)


leftover models on GPU: ['labert_wsd', 'philberta', 'mdeberta']
[before sample embeddings] free=13.23/15.58 GiB  torch_alloc=1.96 GiB  torch_reserved=1.98 GiB
[after offload labert_wsd] free=13.64/15.58 GiB  torch_alloc=1.55 GiB  torch_reserved=1.57 GiB
[after offload philberta] free=14.13/15.58 GiB  torch_alloc=1.05 GiB  torch_reserved=1.08 GiB
[after offload xlmr] free=14.13/15.58 GiB  torch_alloc=1.05 GiB  torch_reserved=1.08 GiB
[after offload mdeberta] free=15.18/15.58 GiB  torch_alloc=0.01 GiB  torch_reserved=0.02 GiB


In [ ]:

# Production export for the full concordances: philberta_l11 + mdeberta_l9.
# (Filename keeps the original `embeddigings` spelling -- notebooks 6/7 read it.)
model_list = [
        ("philberta", model_philberta, tokenizer_philberta),
        ("mdeberta", model_mdeberta, tokenizer_mdeberta),
    ]

LAYERS_FULL = {"philberta": [11], "mdeberta": [9]}

case_studies_terms = ["pathos", "passio"]
for term in case_studies_terms:
    instances = read_hits(f"{term}.parquet")
    embedding_helpers.gpu_mem_report(f"{term}: before encoders")
    for title, model, tokenizer in model_list:
        for layer in LAYERS_FULL.get(title, [11]):
            emb_col_name = f"emb_{title}_l{layer}"
            instances[emb_col_name] = instances.apply(lambda row: embedding_helpers.embed_emlap_instance(
                instance=row,
                tokenizer=tokenizer,
                model=model,
                device="cuda",                # or "cpu"
                layer_idx=layer,
                piece_pooling="mean",
                context_lemmatized=True,
                target_lemmatized=True,
                context_pos_filtered=True)["embedding"], axis=1)
        embedding_helpers.offload(model)      # release before the next encoder
        embedding_helpers.gpu_mem_report(f"{term}: after offload {title}")
    instances.to_parquet("../data/large_files/{}_embeddigings_full.parquet".format(term))
